## [Training a causal language model from scratch](https://huggingface.co/course/chapter7/6?fw=pt)
*This is part 2. Part 1 is **[here](http://localhost:8888/notebooks/7.6-Training_a_causal_language_model_from_scratch_(part_1).ipynb)***.

### Training with 🤗 Accelerate
We've seen how to train a model with the `Trainer`, which can allow for some customization. However, sometimes we want full control over the training loop, or we want to make some exotic changes. In this case  Accelerate is a great choice, and in this section we'll go through the steps to use it to train our model. To make things more interesting, we’ll also add a twist to the training loop.

In [1]:
from components7point6 import * # import required objects from part 1
from IPython.display import HTML
HTML('<iframe width="640" height="360" src="https://www.youtube.com/embed/Hm8_PgVTFuc" allowfullscreen></iframe>')

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/IPython/core/display.py:475: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Since we are mainly interested in sensible autocompletion for the the data science libraries, it makes sense to give more weight to training samples that make more use of these libraries. We can easily identify these examples through the use of keywords such as `plt`, `pd`, `sk`, `fit`, and `predict`, which are the most frequent import names for `matplotlib.pyplot`, `pandas`, and `sklearn` as well as the fit/predict pattern of the latter. If these are each represented as a single token, we can easily check if they occur in the input sequence. Tokens can have a whitespace prefix, so we’ll also check for those versions in the tokenizer vocabulary. To verify that it works, we’ll add one test token which should be split into multiple tokens:

In [2]:
keytoken_ids = []
for keyword in ["plt", "pd", "sk", "fit", "predict", "accelerate"]:
    ids = tokenizer([keyword]).input_ids[0]
    if len(ids) == 1:
        keytoken_ids.append(ids[0])
    else:
        print(f"Keyword has number of tokens ≠ 1:\n{keyword}")

keytoken_ids

Keyword has number of tokens ≠ 1:
accelerate


[8436, 4289, 1201, 2770, 5431]

Great, that seems to work nicely! We can now write a custom loss function that takes the input sequence, the logits, and the key tokens we just selected as inputs. First, we need to align the logits and inputs: the input sequence shifted by one to the right forms the labels, since the next token is the label for the current token. We can achieve this by starting the labels from the second token of the input sequence, since the model does not make a prediction for the first token anyway. Then we cut off the last logit, as we don't have a label for the token that follows the full input sequence. With that we can compute the loss per sample and count the occurrences of all keywords in each sample. Finally, we calculate the weighted average over all samples using the occurrences as weights. Since we don't want to throw away all the samples that have no keywords, we add `1` to the weights:

In [3]:
import torch
from torch.nn import CrossEntropyLoss

def keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0):
    # Shift so that tokens < n predict n
    shift_labels = inputs[..., 1:].contiguous()
    shift_logits = logits[..., :-1, :].contiguous()
    # Calculate per-token loss
    loss_fct = CrossEntropyLoss(reduce=False)
    loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
    # Resize and average loss per sample
    loss_per_sample = loss.view(shift_logits.size(0), shift_logits.size(1)).mean(axis=1)
    # Calculate and scale weighting
    weights = torch.stack([(inputs == kt).float() for kt in keytoken_ids]).sum(
        axis=[0, 2]
    )
    weights = alpha * (1.0 + weights)
    # Calculate weighted average
    weighted_loss = (loss_per_sample * weights).mean()
    return weighted_loss

keytoken_weighted_loss

<function __main__.keytoken_weighted_loss(inputs, logits, keytoken_ids, alpha=1.0)>

Before we can start training with this awesome new loss function, we need to prepare a few things:
- We need dataloaders to load the data in batches.
- We need to set up weight decay parameters.
- From time to time we want to evaluate, so it makes sense to wrap the evaluation code in a function.

Let's start with the dataloaders. We only need to set the dataset's format to `"torch"`, and then we can pass it to a PyTorch `DataLoader` with the appropriate batch size:

In [4]:
from torch.utils.data.dataloader import DataLoader
tokenized_datasets.set_format("torch") # tokenized_datasets["train"].select(range(5000))
train_dataloader= DataLoader(tokenized_datasets["train"], batch_size=8, shuffle=True)
eval_dataloader = DataLoader(tokenized_datasets["valid"], batch_size=8)
train_dataloader, eval_dataloader

(<torch.utils.data.dataloader.DataLoader at 0x71ab131fd250>,
 <torch.utils.data.dataloader.DataLoader at 0x71ab14111c40>)

Next, we group the parameters so that the `optimizer` knows which ones will get an additional weight decay. Usually, all `bias` and `LayerNorm` weights terms are exempt from this; here's how we can do this:

In [5]:
def get_grouped_params(model, no_decay=["bias", "LayerNorm.weight"]):
    params_with_wd, params_without_wd = [], []
    for n, p in model.named_parameters():
        if any(nd in n for nd in no_decay):
            params_without_wd.append(p)
        else:
            params_with_wd.append(p)
    return [
        {"params": params_with_wd, "weight_decay": 0.1},
        {"params": params_without_wd, "weight_decay": 0.0},
    ]

print(f"{str(get_grouped_params(model))[:2000]} ...")

[{'params': [Parameter containing:
tensor([[ 0.0034, -0.0335, -0.0114,  ..., -0.0175,  0.0122,  0.0044],
        [-0.0290,  0.0171,  0.0084,  ..., -0.0012,  0.0373,  0.0184],
        [-0.0084,  0.0695, -0.0231,  ..., -0.0093, -0.0255,  0.0058],
        ...,
        [ 0.0288, -0.0451, -0.0270,  ..., -0.0008,  0.0052, -0.0137],
        [-0.0154,  0.0052,  0.0169,  ..., -0.0212,  0.0156, -0.0401],
        [-0.0208, -0.0126,  0.0213,  ...,  0.0017, -0.0169,  0.0038]],
       requires_grad=True), Parameter containing:
tensor([[ 0.0140,  0.0109, -0.0124,  ...,  0.0427, -0.0328, -0.0388],
        [-0.0302, -0.0171,  0.0026,  ...,  0.0040,  0.0181,  0.0169],
        [-0.0068,  0.0305, -0.0037,  ..., -0.0008, -0.0091, -0.0343],
        ...,
        [ 0.0144, -0.0150, -0.0040,  ..., -0.0293, -0.0240, -0.0274],
        [-0.0246, -0.0169, -0.0459,  ..., -0.0052,  0.0129, -0.0168],
        [ 0.0185,  0.0275,  0.0123,  ...,  0.0090,  0.0231, -0.0276]],
       requires_grad=True), Parameter containin

Since we want to evaluate the model regularly on the validation set during training, let's write a function for that as well. It just runs through the evaluation dataloader and gathers all the losses across processes:

In [6]:
def evaluate(model, dataloader):
    model.eval()
    losses = []
    for step, batch in enumerate(dataloader):
        with torch.no_grad():
            batch = {k: v.to(model.device) for k, v in batch.items()}
            outputs = model(batch["input_ids"], labels=batch["input_ids"])
            loss = outputs.loss
            # Ensure all losses are gathered across devices
            loss = accelerator.gather(loss.repeat(batch["input_ids"].size(0)))
            losses.append(loss)
    # Flatten all losses into a single tensor
    losses = torch.cat(losses)
    loss = torch.mean(losses)
    try:
        perplexity = torch.exp(loss)
    except OverflowError:
        print("OverflowError!")
        perplexity = float("inf")
    return loss.item(), perplexity.item()

evaluate

<function __main__.evaluate(model, dataloader)>

With the `evaluate()` function we can report loss and [perplexity](https://huggingface.co/course/chapter7/3) at regular intervals. Next, we redefine our model to make sure we train from scratch again:

In [7]:
model = GPT2LMHeadModel(config)
type(model)

transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel

We can then define our optimizer, using the function from before to split the parameters for weight decay:

In [8]:
from torch.optim import AdamW
optimizer = AdamW(params=get_grouped_params(model), lr=5e-4)
optimizer

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0005
    maximize: False
    weight_decay: 0.0
)

Now let's prepare the model, optimizer, and dataloaders so we can start training:

In [9]:
from accelerate import Accelerator
accelerator = Accelerator()
model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
    model, optimizer, train_dataloader, eval_dataloader
)
optimizer, train_dataloader, eval_dataloader

(AcceleratedOptimizer (
 Parameter Group 0
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0005
     maximize: False
     weight_decay: 0.1
 
 Parameter Group 1
     amsgrad: False
     betas: (0.9, 0.999)
     capturable: False
     differentiable: False
     eps: 1e-08
     foreach: None
     fused: None
     lr: 0.0005
     maximize: False
     weight_decay: 0.0
 ),
 <torch.utils.data.dataloader.DataLoader at 0x71ac1e5ff260>)

> 🚨 <font color="darkgreen">If you're training on a TPU, you'll need to move all the code starting at the cell above into a dedicated training function. See [Chapter 3](https://huggingface.co/course/chapter3) for more details.</font>

Now that we have sent our `train_dataloader` to `accelerator.prepare()`, we can use its length to compute the number of training steps. Remember that we should always do this after preparing the dataloader, as that method will change its length. We use a classic linear schedule from the learning rate to 0:

In [10]:
from transformers import get_scheduler
num_train_epochs = 3
num_update_steps_per_epoch = len(train_dataloader)
num_training_steps = num_train_epochs * num_update_steps_per_epoch
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=1_000,
    num_training_steps=num_training_steps,
)
lr_scheduler

Lastly, to push our model to the Hub, we will need to create a `Repository` object in a working folder. First log in to the Hugging Face Hub, if you aren't logged in already. We'll determine the repository name from the model ID we want to give our model (feel free to replace the `repo_name` with your own choice; it just needs to contain your username, which is what the function `get_full_repo_name()` does):

In [11]:
from huggingface_hub import Repository, get_full_repo_name, create_repo
model_name = "codeparrot-model-accelerate"
repo_name = get_full_repo_name(model_name)
repo_name

'mdroth/codeparrot-model-accelerate'

Then we can clone that repository in a local folder. If it already exists, this local folder should be an existing clone of the repository we are working with:

In [12]:
output_dir = "sections/section_7/logs/codeparrot-ds-accelerate"
# try to get repo
try:
    repo = Repository(output_dir, clone_from=repo_name)
    repo_message = f"The '{repo_name}' repo has already been created."
# otherwise, create repo
except:
    repo = create_repo(repo_name)
    repo_message = f"The '{repo_name}' repo has just been created."
print(repo_message)

/home/matthias/Desktop/MachineLearning/Huggingface-course/env/lib/python3.12/site-packages/huggingface_hub/utils/_deprecation.py:131: FutureWarning: 'Repository' (from 'huggingface_hub.repository') is deprecated and will be removed from version '1.0'. Please prefer the http-based alternatives instead. Given its large adoption in legacy code, the complete removal is only planned on next major release.
For more details, please read https://huggingface.co/docs/huggingface_hub/concepts/git_vs_http.
  warnings.warn(warning_message, FutureWarning)
/home/matthias/Desktop/MachineLearning/Huggingface-course/sections/section_7/logs/codeparrot-ds-accelerate is already a clone of https://huggingface.co/mdroth/codeparrot-model-accelerate. Make sure you pull the latest changes with `repo.git_pull()`.


The 'mdroth/codeparrot-model-accelerate' repo has already been created.


We can now upload anything we save in `output_dir` by calling the `repo.push_to_hub()` method. This will help us upload the intermediate models at the end of each epoch.

Before we train, let's run a quick test to see if the evaluation function works properly:

In [13]:
evaluate(model=model, dataloader=eval_dataloader)

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


(11.035737991333008, 62052.6171875)

Those are very high values for loss and perplexity, but that's not surprising as we haven't trained the model yet. With that, we have everything prepared to write the core part of the training script: the training loop. In the training loop we iterate over the dataloader and pass the batches to the model. With the logits, we can then evaluate our custom loss function. We scale the loss by the number of gradient accumulation steps so as not to create larger losses when aggregating more steps. Before we optimize, we also clip the gradients for better convergence. Finally, every few steps we evaluate the model on the evaluation set with our new `evaluate()` function:

In [ ]:
from tqdm.notebook import tqdm
from torch.nn.utils import clip_grad_norm_
# training config
num_epochs = 2
# training loop
for epoch in range(num_epochs):
    model.train()
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    total_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc="Training", leave=False)
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(model.device)
        # use automatically shifted input_ids as labels for causal LM (next token prediction)
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        # gradient clipping (useful for training stability)
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Average training loss after epoch {epoch + 1}: {avg_train_loss:.4f}")
    # evaluation after epoch
    model.eval()
    eval_loss, perplexity = evaluate(model, eval_dataloader)
    print(f"Eval loss: {eval_loss:.4f} | Perplexity: {perplexity:.2f}")


Epoch 1/2


Training:   0%|          | 0/18218 [00:00<?, ?it/s]

And that's it — you now have your own custom training loop for causal language models such as GPT-2 that you can further customize to your needs.

> ✏️ Try it out! <font color="darkgreen">Either create your own custom loss function tailored to your use case, or add another custom step into the training loop.</font>

In [ ]:
###########################################
# trying it out 3                         #
# use another custom step: counting flops #
###########################################
import numpy as np
from torch.nn.utils import clip_grad_norm_
from deepspeed.profiling.flops_profiler import FlopsProfiler
prof = FlopsProfiler(model) # deepspeed profiler
flops_list = []
# training loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    total_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc="Training", leave=False)
    model.train()
    # training
    prof.start_profile() # start profiling
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(model.device)
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
    prof.stop_profile() # stop profiling
    total_flops = prof.get_total_flops()
    flops_list.append(total_flops)
    # validation
    model.eval()
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Average training loss after epoch {epoch + 1}: {avg_train_loss:.4f}")
    # evaluation
    eval_loss, perplexity = evaluate(model, eval_dataloader)
    print(f"Eval loss: {eval_loss:.4f} | Perplexity: {perplexity:.2f}")
prof.end_profile() # end profiling
mean_flops = round(np.array(flops_list).mean())
print(f"\nflops list: {flops_list}\nflops mean: {mean_flops}")

> ✏️ Try it out! <font color="darkgreen">When running long training experiments it's a good idea to log important metrics using tools such as TensorBoard or Weights & Biases. Add proper logging to the training loop so you can always check how the training is going.</font>

In [ ]:
############################################
# trying it out 4: Add logging with wandb! #
############################################
import os
import wandb
from dotenv import load_dotenv
load_dotenv()
# initialize wandb
wandb.login(key=os.environ["WANDB_TOKEN"])
os.environ["WANDB_MODE"] = "online"
print(os.environ["WANDB_MODE"])
wandb.init(
    project = "gpt2-training",
    name = "7.6-part_2-try_4",
    config = {
        "epochs": num_epochs,
        "batch_size": train_dataloader.batch_size,
        "model": "gpt2",
        "lr": lr_scheduler.get_last_lr()[0]
    }
)
# instantiate flops profiler
prof = FlopsProfiler(model)
flops_list = []
# training loop
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    total_loss = 0.0
    model.train()
    progress_bar = tqdm(train_dataloader, desc="Training", leave=False)
    # start counting flops
    prof.start_profile()
    for step, batch in enumerate(progress_bar):
        input_ids = batch["input_ids"].to(model.device)
        outputs = model(input_ids=input_ids, labels=input_ids)
        loss = outputs.loss
        loss.backward()
        clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())
        wandb.log({
            "train/loss_step": loss.item(),
            "train/lr": lr_scheduler.get_last_lr()[0],
            "step": epoch * len(train_dataloader) + step
        })
    # stop counting flops
    prof.stop_profile()
    total_flops = prof.get_total_flops()
    flops_list.append(total_flops)
    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Average training loss after epoch {epoch + 1}: {avg_train_loss:.4f}")
    model.eval()
    eval_loss, perplexity = evaluate(model, eval_dataloader)
    print(f"Eval loss: {eval_loss:.4f} | Perplexity: {perplexity:.2f}\n")
    wandb.log({
        "epoch": epoch + 1,
        "train/loss_epoch": avg_train_loss,
        "eval/loss": eval_loss,
        "eval/perplexity": perplexity,
        "flops": total_flops,
    })
# get flops
prof.end_profile()
mean_flops = round(np.array(flops_list).mean())
print(f"\nflops list: {flops_list}\nflops mean: {mean_flops}")
# final wandb log
wandb.log({"flops_mean": mean_flops})
wandb.finish()
os.environ["WANDB_MODE"] = "offline"
print(os.environ["WANDB_MODE"])

$\checkmark$